In [1]:
pip install pandas numpy faker mysql-connector-python

   ---------------------------------------- 2.0/2.0 MB 1.6 MB/s eta 0:00:00
   ---------------------------------------- 16.4/16.4 MB 2.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import numpy as np
from faker import Faker
import mysql.connector

fake = Faker()

print("Random Name:", fake.name())
print("Random Number:", np.random.randint(1, 100))

df = pd.DataFrame({
    "Name": [fake.name() for _ in range(5)],
    "Age": np.random.randint(20, 50, 5)
})

print(df)

Random Name: Julia Richards
Random Number: 20
                 Name  Age
0      Clifford Bates   32
1  Barbara Williamson   25
2     Michael Fuentes   42
3         Justin Ball   21
4        Lynn Jackson   34


In [7]:
pip install sqlalchemy openpyxl jupyter matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
pip install rapidfuzz python-dateutil

   ---------------------------------------- 1.6/1.6 MB 864.4 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
# Script 1/5 — fact_orders
#Generates 5,000 rows for the Retail & E-Commerce dataset.
#Injects: nulls, duplicates, mixed date formats, outliers,
#         join challenges (orphaned FKs, wrong regions),
#         and validation violations (ship < order, zero-revenue, dup IDs).

In [11]:
import os, random
import pandas as pd
import numpy as np
from faker import Faker

In [12]:
fake = Faker()

In [13]:
random.seed(42)
np.random.seed(42)
Faker.seed(42)

In [14]:
os.makedirs("data/raw_1", exist_ok=True)

In [18]:
# constants

In [15]:
N=5000

In [17]:
CUSTOMER_IDS = [f"CUST-{str(i).zfill(3)}"for i in range (1, 1001)]
PRODUCT_IDS   = [f"PROD-{str(i).zfill(3)}" for i in range(1, 201)]
LOCATION_IDS  = [f"LOC-{str(i).zfill(3)}" for i in range(1, 101)]
REP_IDS       = [f"REP-{str(i).zfill(3)}" for i in range(1, 76)]
STATUSES      = ["Completed", "Pending", "Returned", "Cancelled"]
STATUS_WEIGHTS = [0.60, 0.20, 0.12, 0.08]
REGIONS        = ["North", "South", "East", "West", "Central"]

In [19]:
# Helper: random date between two dates

In [20]:
def rand_date (start, end):
    delta = (end - start).days
    return start + pd.Timedelta(days = random.randint(0,delta))

In [21]:
START = pd.Timestamp("2023-01-01")
END = pd.Timestamp("2024-12-31")

In [22]:
# Base generation

In [26]:
records = []
for i in range(1, N+1):
    order_date = rand_date(START, END)
    ship_date = order_date + pd.Timedelta(days=random.randint(1, 14))
    qty = random.randint(1, 50)
    unit_price = round(random.uniform(5, 500), 2)
    discount = round(random.uniform(0, 0.40), 4)
    revenue = round(qty * unit_price * (1 - discount), 2)
    cost = round(revenue * random.uniform(0.40, 0.80), 2)
    status = random.choices(STATUSES, STATUS_WEIGHTS)[0]
    region = random.choice(REGIONS)
    
    records.append({
        "order_id":     f"ORD-{str(i).zfill(5)}",
        "customer_id":  random.choice(CUSTOMER_IDS),
        "product_id":   random.choice(PRODUCT_IDS),
        "location_id":  random.choice(LOCATION_IDS),
        "rep_id":       random.choice(REP_IDS),
        "order_date":   order_date,
        "ship_date":    ship_date,
        "quantity":     qty,
        "unit_price":   unit_price,
        "revenue":      revenue,
        "cost":         cost,
        "discount_pct": discount,
        "status":       status,
        "region":       region,
    })

In [27]:
df = pd.DataFrame(records)

In [28]:
# PART 2 - Data Quality injections

In [29]:
# a) 10-15% nulls across 4 columns

In [30]:
for col, rate in [("rep_id", 0.12), ("discount_pct", 0.10),
                 ("location_id", 0.11), ("region", 0.13)]:
    null_idx = df.sample(frac=rate, random_state=42).index
    df.loc[null_idx, col] = np.nan

In [31]:
# b) 3-5% near duplicates

In [32]:
dup_idx = df.sample(frac=0.04, random_state =7).index
dups = df.loc[dup_idx].copy()

In [33]:
# mutate order_id slightly so they look like near-dupes

In [34]:
dups["order_id"] = dups["order_id"].str.replace("ORD-", "ord-")
df = pd.concat([df, dups], ignore_index= True)

In [35]:
# c) Mixed date formats

In [39]:
def mixed_date_format(dt, style):
    if pd.isna(dt):
        return dt
    
    ts = pd.Timestamp(dt)
    
    if style == 0:
        return ts.strftime("%m/%d/%Y")   # MM/DD/YYYY
    elif style == 1:
        return ts.strftime("%Y-%m-%d")   # YYYY-MM-DD
    else:
        return ts.strftime("%d-%b-%Y")   # DD-Mon-YYYY


styles_order = np.random.choice([0, 1, 2], size=len(df))
styles_ship = np.random.choice([0, 1, 2], size=len(df))

df["order_date"] = [mixed_date_format(d, s) for d, s in zip(df["order_date"], styles_order)]
df["ship_date"]  = [mixed_date_format(d, s) for d, s in zip(df["ship_date"], styles_ship)]

In [40]:
# e) Mixed currency strings for unit price

In [42]:
def mixed_currency(val, style):
    if pd.isna(val):
        return val
    if style ==0:
        return f"${val:,.2f}"
    elif style == 1:
        return str(int(val))
    else:
        return f"{val:,.2f}".replace("$", "")

price_styles = np.random.choice([0, 1, 2], size=len(df))
df["unit_price"] = [mixed_currency(v, s) for v, s in zip(df["unit_price"], price_styles)]

In [43]:
# g) Inconsistent customer_id prefixes

In [44]:
prefix_idx = df.sample(frac=0.06, random_state=99).index
df.loc[prefix_idx, "customer_id"] = (
    df.loc[prefix_idx, "customer_id"]
        .str.replace("CUST-", "C", regex = False)
        .str.lstrip("0")
)

In [45]:
# h) outliers

In [47]:
# Negative revenue (1%)

In [48]:
neg_idx = df.sample(frac=0.01, random_state =11).index
df.loc[neg_idx, "revenue"] = df.loc[neg_idx, "revenue"].apply(lambda x: -abs(float(str(x).replace("$", "").replace (",","")))if pd.notna(x)else x)

In [49]:
# Future ship dates (0.5%)

In [50]:
future_idx = df.sample(frac=0.005, random_state=22).index
df.loc[future_idx, "ship_date"] = "2026-07-15"

In [51]:
# Zero quantity comleted orders (0.5%)

In [52]:
zq_idx = df[df["status"]=="Completed"].sample(frac=0.007, random_state=33).index
df.loc[zq_idx, "quantity"]=0

In [53]:
# PART 3 - Join Challenges

In [54]:
# a) 4% orphaned customer_id (IDs that won't exist in dim_customers)

In [55]:
orphan_idx = df.sample(frac = 0.04, random_state=55).index
df.loc[orphan_idx, "custmer_id"] = [f"CUST-{random.randint(9000, 9999)}" for _ in orphan_idx]

In [57]:
# PART 5 - Validation Violation

In [58]:
# a) 5 rows ship_date < order_date

In [59]:
bad_ship_idx = df.sample(5, random_state=77).index
df.loc[bad_ship_idx, "ship_date"] = "2022-12-01"  # clearly before any order_date
df.loc[bad_ship_idx, "order_date"] = "2023-06-15"

In [61]:
# b) 3 rows revenue = 0 where status = Completed 

In [62]:
zero_rev_idx = df[df["status"]=="Completed"].sample(3, random_state = 88).index
df.loc[zero_rev_idx, "revenue"] = 0

In [63]:
# c) Duplicate order_id pairs with different revenue

In [65]:
dup_pairs = df.sample(4, random_state=66).copy()
dup_pairs["revenue"] = dup_pairs["revenue"].apply(
    lambda x: round(float(str(x).replace("$","").replace(",",""))*1.15, 2)
    if pd.notna(x) else x
)

In [66]:
# Keep same order_id but restore ORD- prefix

In [67]:
dup_pairs["order_id"] = df.sample(4, random_state=66)["order_id"].values
df = pd.concat([df, dup_pairs], ignore_index = True)

In [68]:
# d) 6 rows region mismatch vs dim_customers

In [69]:
mismatch_idx = df.sample(6, random_state=44).index
other_regions = ["North", "South", "East", "West", "Central"]
df.loc[mismatch_idx, "region"] = [
    random.choice([r for r in other_regions if r != str(df.loc[i, "region"])])
    for i in mismatch_idx
]
df.loc[mismatch_idx, "_region_mismatch_flag"]= True # will be dropped before BI

In [70]:
# Final shuffle and save

In [71]:
df = df.sample(frac = 1, random_state=42).reset_index(drop=True)
out_path = "data/raw_1/fact_orders.csv"
df.to_csv(out_path, index = False)
print(f"✅ fact_orders saved → {out_path}  |  shape: {df.shape}")
print(df.dtypes)
print(df.head(3))

✅ fact_orders saved → data/raw_1/fact_orders.csv  |  shape: (5204, 16)
order_id                  object
customer_id               object
product_id                object
location_id               object
rep_id                    object
order_date                object
ship_date                 object
quantity                   int64
unit_price                object
revenue                  float64
cost                     float64
discount_pct             float64
status                    object
region                    object
custmer_id                object
_region_mismatch_flag     object
dtype: object
    order_id customer_id product_id location_id   rep_id  order_date  \
0  ORD-00469    CUST-292   PROD-167         NaN      NaN  03/23/2023   
1  ORD-00297    CUST-470   PROD-192         NaN      NaN  2023-05-26   
2  ORD-04613    CUST-041   PROD-153     LOC-007  REP-051  08/16/2024   

     ship_date  quantity unit_price  revenue     cost  discount_pct  \
0  30-Mar-2023        41   